# Лабораторная работа № 2.2
---
«Построение модели probit-регрессии»

Рязанцев Георгий Борисович

Данные `FinCond.xlsx` — финансовое состояние 62 сельскохозяйственных организаций. Переменная $Y$ фиктивная: 1 — организация банкрот, 0 — не банкрот; $X_1$ и $X_2$ — показатели финансового состояния.

## 1. Загрузка библиотек и данных

In [1]:
import math
import pandas as pd
import numpy as np
from statsmodels.discrete.discrete_model import Probit
import statsmodels.api as sm

In [2]:
data = pd.read_excel('FinCond.xlsx')
print(data.head())

          Х1        Х2  Y
0  33.599802  0.291216  0
1   9.922273  0.348461  0
2  17.576723  0.323312  0
3  20.616918  0.148271  0
4  24.919599  0.245431  0


## 2. Описательная статистика

In [3]:
print(data.describe())

              Х1         Х2          Y
count  62.000000  62.000000  62.000000
mean    9.669272  -0.342726   0.500000
std    11.462796   1.516815   0.504082
min     0.000000  -9.642577   0.000000
25%     0.736261  -0.279469   0.000000
50%     5.430789   0.157197   0.500000
75%    15.294005   0.349554   1.000000
max    43.408164   0.730349   1.000000


In [4]:
# значения с рис. 2.15 методички
book_describe = pd.DataFrame(
    {
        data.columns[0]: [62, 9.669272, 11.462796, 0.000000, 0.736261, 5.430789, 15.294005, 43.408164],
        data.columns[1]: [62, -0.342726, 1.516815, -9.642577, -0.279469, 0.157197, 0.349554, 0.730349],
        data.columns[2]: [62, 0.500000, 0.504082, 0.000000, 0.000000, 0.500000, 1.000000, 1.000000],
    },
    index=['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max'],
)
diff = (data.describe() - book_describe).abs().max().max()
print('Максимальное отклонение реальных значений от рис. 2.15:', diff)
print('Совпадает с точностью до 6 знаков (как на рисунке):', bool(diff < 5e-7))
print()
print(data['Y'].value_counts().sort_index().to_string())
print('Организаций с X2 < -2:', int((data[data.columns[1]] < -2).sum()))

Максимальное отклонение реальных значений от рис. 2.15: 4.906638331192426e-07
Совпадает с точностью до 6 знаков (как на рисунке): True

Y
0    31
1    31
Организаций с X2 < -2: 4


**Результат совпадает с рис. 2.15 до шестого знака.** Всего 62 организации, 31 банкрот и 31 не банкрот, то есть, как сказано в методичке, совокупность делится ровно пополам (среднее $Y = 0{,}5$).

Дополнительное наблюдение, которое понадобится ниже при интерпретации. У $X_2$ среднее ($-0{,}343$) намного ниже медианы ($0{,}157$): среднее тянут вниз единичные очень малые значения (минимум $-9{,}64$, четыре организации с $X_2 < -2$). Поэтому «средняя» по $X_2$ организация нетипична для основной массы выборки. У $X_1$ среднее ($9{,}67$) выше медианы ($5{,}43$) — распределение скошено вправо.

## 3. Построение probit-модели

In [5]:
Y = data["Y"]
X = data[["Х1", "Х2"]]
X = sm.add_constant(X)

In [6]:
model = Probit(Y, X.astype(float))
probit_model = model.fit()
print(probit_model.summary())

Optimization terminated successfully.
         Current function value: 0.088668
         Iterations 11
                          Probit Regression Results                           
Dep. Variable:                      Y   No. Observations:                   62
Model:                         Probit   Df Residuals:                       59
Method:                           MLE   Df Model:                            2
Date:                Tue, 24 Nov 2026   Pseudo R-squ.:                  0.8721
Time:                        19:40:18   Log-Likelihood:                -5.4974
converged:                       True   LL-Null:                       -42.975
Covariance Type:            nonrobust   LLR p-value:                 5.292e-17
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.6424      0.970      2.725      0.006       0.742       4.543
Х1            -0.3658      0

In [7]:
# сверка с рис. 2.16 по всем показателям summary() (coef, std err, z, p, ДИ, LL, LL-Null, LLR p, Pseudo R², N)
# допуск — единица последнего знака, показанного в методичке; таблица сравнения свёрнута в одну строку вывода
book = {  # рис. 2.16: coef, std err, z, P>|z|, [0.025, 0.975]
    'const': (2.6424, 0.970, 2.725, 0.006, 0.742, 4.543),
    'Х1': (-0.3658, 0.154, -2.377, 0.017, -0.667, -0.064),
    'Х2': (-5.3241, 2.105, -2.529, 0.011, -9.450, -1.198),
}
ci = probit_model.conf_int()
ok = True
for name, (c, se, z_, p_, lo, hi) in book.items():
    ok &= (abs(probit_model.params[name] - c) <= 1e-4 and abs(probit_model.bse[name] - se) <= 1e-3
           and abs(probit_model.tvalues[name] - z_) <= 1e-3 and abs(probit_model.pvalues[name] - p_) <= 1e-3
           and abs(ci.loc[name].iloc[0] - lo) <= 1e-3 and abs(ci.loc[name].iloc[1] - hi) <= 1e-3)
ok &= (abs(probit_model.llf + 5.4974) <= 1e-4 and abs(probit_model.llnull + 42.975) <= 1e-3
       and abs(probit_model.llr_pvalue - 5.292e-17) <= 1e-20 and abs(probit_model.prsquared - 0.8721) <= 1e-4
       and int(probit_model.nobs) == 62)
print('Совпадает с рис. 2.16:', ok)

Совпадает с рис. 2.16: True


In [8]:
# качество «в лоб»: доля верно классифицированных организаций и насколько расчётные вероятности близки к 0 или 1
p_fit = np.asarray(probit_model.predict(X.astype(float)))
print('Верно классифицировано (порог 0.5):', int(((p_fit > 0.5).astype(int) == Y.to_numpy()).sum()), 'из', len(Y))
print('Организаций с расчётной вероятностью банкротства в диапазоне 0.05–0.95:', int(((p_fit > 0.05) & (p_fit < 0.95)).sum()), 'из', len(Y))

Верно классифицировано (порог 0.5): 60 из 62
Организаций с расчётной вероятностью банкротства в диапазоне 0.05–0.95: 9 из 62


**Результаты совпадают с рис. 2.16** (все показатели — см. вывод выше). Отличия реального вывода только технические: строки `Date`/`Time`, служебные строки оптимизатора (`Optimization terminated successfully`, `Current function value: 0.088668` $= -LL/N$) и примечание под таблицей (обрезано на рис. 2.16).

**Уравнение:** $Z = 2{,}6424 - 0{,}3658\,X_1 - 5{,}3241\,X_2$, $\Pr(Y=1) = \Phi(Z)$.

**Интерпретация.**
- Все три коэффициента значимы на 5 % ($p = 0{,}006$; $0{,}017$; $0{,}011$), модель в целом значима ($LLR\ p = 5{,}29\cdot10^{-17}$) — подтверждает вывод методички о достоверности модели.
- Знаки при $X_1$ и $X_2$ отрицательные: чем лучше финансовое состояние, тем ниже вероятность банкротства.
- Как и сказано в методичке, сами коэффициенты probit не интерпретируются напрямую (это не прирост вероятности) — они нужны для расчёта вероятностей и предельных эффектов.
- **Про Pseudo R².** Методичка пишет «близок к 1»; реальное значение $0{,}8721$ ($R^2$ Макфаддена). Для моделей бинарного выбора это очень много (обычно уже $0{,}2$–$0{,}4$ — отличное качество): классы почти полностью разделены (см. ячейку выше — 60 верных из 62, лишь у 9 организаций вероятность в диапазоне 0,05–0,95). Оценка внутривыборочная, поэтому на новых данных качество будет ниже.

**Примечание `Possibly complete quasi-separation` (доля 0,56)** — не ошибка: оценка сошлась, коэффициенты конечны и равны рис. 2.16, полного разделения классов нет. Срабатывает из-за большой доли почти безошибочно предсказанных наблюдений (у logit та же доля 0,42). Согласуется с выводом о почти полном разделении: оценки опираются на 9 пограничных организаций и могут быть чувствительны к составу выборки.

## 4. Вероятность банкротства «средней» организации

In [9]:
# расчёт по методичке: z в точке средних значений, затем формула (2.3) с π ≈ 3.14 и e ≈ 2.72
z = 2.6424-0.3658*9.669272-5.3241*(-0.342726)
pb = (1/(math.sqrt(2*3.14)))*(2.72)**(-0.5*(z*z))
print(round(z, 2))
print(pb)

0.93
0.25885429182896547


In [10]:
from scipy.stats import norm

x_mean = X.astype(float).mean().to_numpy()                  # [1, среднее X1, среднее X2]
z_real = float(probit_model.params.to_numpy() @ x_mean)      # Z в точке средних значений (без округления коэффициентов)

density = norm.pdf(z_real)   # φ(z): то, что считает формула (2.3) методички
prob = norm.cdf(z_real)      # Φ(z): вероятность по формуле (2.1) probit-модели
prob_model = float(np.asarray(probit_model.predict(x_mean.reshape(1, -1)))[0])   # то же средствами statsmodels

print(f'Z (реальное значение)                         = {z_real:.4f}')
print(f'φ(Z), формула (2.3) методички                = {density:.6f}')
print(f'Φ(Z) = Pr(Y=1), вероятность банкротства      = {prob:.4f}')
print(f'probit_model.predict в точке средних значений = {prob_model:.4f}')

Z (реальное значение)                         = 0.9302
φ(Z), формула (2.3) методички                = 0.258843
Φ(Z) = Pr(Y=1), вероятность банкротства      = 0.8239
probit_model.predict в точке средних значений = 0.8239


**Что совпадает.** $z = 0{,}93$ и `pb = 0.25885429…` в точности равны рис. 2.17; разница с «точным» $\varphi(Z) = 0{,}258843$ — только от округлений в коде методички ($\pi \approx 3{,}14$, $e \approx 2{,}72$, коэффициенты с четырьмя знаками).

**Расхождение по смыслу: 26 % — не вероятность банкротства.** Формула (2.3) методички $\frac{1}{\sqrt{2\pi}}e^{-z^2/2}$ — это **плотность** $\varphi(z)$, а вероятность в probit по формуле (2.1) самой методички равна $\Phi(Z)$. Быстрая проверка без расчётов: при $z = 0{,}93 > 0$ вероятность обязана превышать 50 %, а плотность никогда не бывает больше $0{,}399$ и вероятностью быть не может. Реальное значение $\Phi(z) = 0{,}8239$ подтверждено `predict` (ячейка выше).

**Вывод:** вероятность банкротства организации со средними $X_1 = 9{,}67$, $X_2 = -0{,}34$ — примерно **82 %**. Учтите, что «средняя» организация нетипична из-за выбросов по $X_2$ (медиана $0{,}157$ против среднего $-0{,}343$, см. раздел 2).

## 5. Предельные эффекты факторов

In [11]:
# расчёт по методичке: предельный эффект X1 = φ(z) · b1
px1 = (1/(math.sqrt(2*3.14)))*(2.72)**(-0.5*(z*z))*(-0.3658)
print(round(px1, 4))

-0.0947


In [12]:
b = probit_model.params.to_numpy()
me_mean = density * b[1:]          # предельные эффекты в точке средних значений: φ(Z)·b_j
print('Предельный эффект X1 (в точке средних):', round(me_mean[0], 4))
print('Предельный эффект X2 (в точке средних):', round(me_mean[1], 4))
print()
print(probit_model.get_margeff(at='mean').summary())

Предельный эффект X1 (в точке средних): -0.0947
Предельный эффект X2 (в точке средних): -1.3781

       Probit Marginal Effects       
Dep. Variable:                      Y
Method:                          dydx
At:                              mean
                dy/dx    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Х1            -0.0947      0.113     -0.839      0.401      -0.316       0.126
Х2            -1.3781      1.157     -1.192      0.233      -3.645       0.889


In [13]:
# точное изменение вероятности при конкретном изменении фактора (из точки средних значений)
d_x1 = norm.cdf(z_real + b[1]) - prob          # X1 + 1
d_x2 = norm.cdf(z_real + 0.1 * b[2]) - prob    # X2 + 0.1
print(f'Вероятность при X1 + 1   : {prob:.4f} -> {prob + d_x1:.4f}  (изменение {d_x1:+.4f})')
print(f'Вероятность при X2 + 0.1 : {prob:.4f} -> {prob + d_x2:.4f}  (изменение {d_x2:+.4f})')

# средний предельный эффект по всем 62 организациям
print()
print(probit_model.get_margeff(at='overall').summary())

# порядок std err предельного эффекта X1 (~0.11) подтверждён отдельным расчётом методом дельты (не показан)

Вероятность при X1 + 1   : 0.8239 -> 0.7137  (изменение -0.1101)
Вероятность при X2 + 0.1 : 0.8239 -> 0.6546  (изменение -0.1693)

       Probit Marginal Effects       
Dep. Variable:                      Y
Method:                          dydx
At:                           overall
                dy/dx    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Х1            -0.0175      0.005     -3.768      0.000      -0.027      -0.008
Х2            -0.2548      0.043     -5.860      0.000      -0.340      -0.170


**Что совпадает.** Предельный эффект $X_1 = \varphi(z)\cdot b_1 = -0{,}0947$ равен рис. 2.18; `get_margeff(at='mean')` даёт то же значение. Здесь формула (2.3) применена правильно: именно для предельных эффектов нужна плотность $\varphi(z)$ (в отличие от пункта 4).

**Уточнения к интерпретации.**
- Методичка: «вероятность сократится на 9,5 %». Правильно — на **9,5 процентных пункта** (с $82{,}4\%$ до $\approx 72{,}9\%$); в относительных процентах было бы ~11,5 %.
- Предельный эффект — производная в точке (приближение для малых изменений). Точное изменение при $X_1 + 1$ равно $-0{,}1101$ ($-11{,}0$ п.п.): приближение занижено по модулю, т.к. при уменьшении $z$ (с $0{,}93$ до $0{,}56$) плотность растёт и кривая становится круче.
- Эффект $X_2 = -1{,}378$ «на единицу» по модулю больше 1 и как изменение вероятности смысла не имеет; разброс $X_2$ мал (QI $-0{,}28\ldots0{,}35$), поэтому считаем для $+0{,}1$: снижение на $16{,}9$ п.п.
- Эффекты **в точке средних** ($-0{,}0947$; $-1{,}378$) заметно больше **средних по выборке** ($-0{,}0175$; $-0{,}2548$): большинство организаций (53 из 62) лежат в «хвостах», где вероятность близка к 0 или 1 и $\varphi \approx 0$, а точка средних попадает в зону максимальной чувствительности. Значит $-9{,}5$ п.п. относится к «средней» организации, а не ко всей выборке.
- **Значимость.** В выводе `get_margeff` (столбцы `std err`, `z`, `P>|z|` методичкой не рассматриваются): эффекты в точке средних **статистически незначимы** ($X_1$: $p = 0{,}401$, ДИ включает ноль; $X_2$: $p = 0{,}233$), хотя сами коэффициенты значимы. Причина: эффект в точке зависит от всех трёх коэффициентов с относительными ошибками 37–42 %, а при почти полном разделении классов такая оценка неточна. Средние по выборке эффекты значимы ($p < 0{,}001$). Поэтому «9,5 п.п.» из методички — точечная оценка с широким интервалом неопределённости; надёжнее опираться на знак коэффициента и средний эффект $-1{,}75$ п.п.

## Самостоятельная работа

### 1. Логит-регрессия

Логит отличается от пробита только функцией связи (логистической вместо $\Phi$) — строим и сравниваем.

In [14]:
from statsmodels.discrete.discrete_model import Logit

logit_model = Logit(Y, X.astype(float)).fit()
print(logit_model.summary())

Optimization terminated successfully.
         Current function value: 0.090223
         Iterations 11
                           Logit Regression Results                           
Dep. Variable:                      Y   No. Observations:                   62
Model:                          Logit   Df Residuals:                       59
Method:                           MLE   Df Model:                            2
Date:                Tue, 24 Nov 2026   Pseudo R-squ.:                  0.8698
Time:                        19:40:18   Log-Likelihood:                -5.5938
converged:                       True   LL-Null:                       -42.975
Covariance Type:            nonrobust   LLR p-value:                 5.828e-17
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.8180      1.954      2.466      0.014       0.988       8.648
Х1            -0.6672      0

In [15]:
coef = pd.DataFrame({'probit': probit_model.params, 'logit': logit_model.params})
coef['logit / probit'] = coef['logit'] / coef['probit']
print(coef.round(4).to_string())
print('π/√3 =', round(np.pi / np.sqrt(3), 3))
print()

odds = np.exp(logit_model.params)
print('Отношения шансов exp(b):')
print(odds.map('{:.4g}'.format).to_string())
print()
print('Вероятность банкротства «средней» организации, logit:', round(float(np.asarray(logit_model.predict(x_mean.reshape(1, -1)))[0]), 4))
print('Вероятность банкротства «средней» организации, probit:', round(prob_model, 4))
print()
print(logit_model.get_margeff(at='mean').summary())

       probit   logit  logit / probit
const  2.6424  4.8180          1.8234
Х1    -0.3658 -0.6672          1.8241
Х2    -5.3241 -9.4458          1.7742
π/√3 = 1.814

Отношения шансов exp(b):
const        123.7
Х1          0.5131
Х2       7.902e-05

Вероятность банкротства «средней» организации, logit: 0.8325
Вероятность банкротства «средней» организации, probit: 0.8239

        Logit Marginal Effects       
Dep. Variable:                      Y
Method:                          dydx
At:                              mean
                dy/dx    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Х1            -0.0930      0.125     -0.743      0.458      -0.339       0.152
Х2            -1.3169      1.350     -0.975      0.329      -3.963       1.329


**Интерпретация.**
- Знаки и значимость те же, что у probit: $b_{X_1} = -0{,}667$, $b_{X_2} = -9{,}446$, все коэффициенты значимы на 5 %.
- Коэффициенты logit примерно в 1,8 раза больше probit (1,82; 1,82; 1,77). Это масштаб, а не различие моделей: стандартное отклонение логистического распределения $\pi/\sqrt{3} \approx 1{,}81$, поэтому коэффициенты напрямую не сравниваются.
- В отличие от probit, коэффициенты logit читаются через шансы: $e^{-0{,}667} = 0{,}513$ — рост $X_1$ на единицу уменьшает шансы банкротства примерно вдвое (на 49 %); для $X_2$: $e^{-9{,}446} \approx 8\cdot10^{-5}$.
- Прогнозы практически совпадают: вероятность «средней» организации $83{,}3\%$ (logit) против $82{,}4\%$ (probit); предельный эффект $X_1$ в точке средних $-0{,}0930$ против $-0{,}0947$ — и, как у probit, статистически незначим ($p = 0{,}458$; для $X_2$: $-1{,}317$, $p = 0{,}329$).

### 2. Параметры качества probit- и logit-моделей

In [16]:
from sklearn.metrics import confusion_matrix, roc_auc_score, brier_score_loss

def quality(res):
    p = np.asarray(res.predict(X.astype(float)))
    pred = (p > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(Y.to_numpy(), pred).ravel()
    return {
        'Log-Likelihood': res.llf,
        'LL-Null': res.llnull,
        'LLR (χ², 2 степени свободы)': res.llr,
        'p-value LLR': res.llr_pvalue,
        'Pseudo R² (McFadden)': res.prsquared,
        'AIC': res.aic,
        'BIC': res.bic,
        'Верно классифицировано, доля': (tp + tn) / len(Y),
        'Верно классифицировано, шт. из 62': tp + tn,
        'Банкротов, признанных небанкротами': fn,
        'Небанкротов, признанных банкротами': fp,
        'ROC AUC': roc_auc_score(Y.to_numpy(), p),
        'Brier score (средний квадрат ошибки вероятности)': brier_score_loss(Y.to_numpy(), p),
    }

table = pd.DataFrame({'Probit': quality(probit_model), 'Logit': quality(logit_model)})
print(table.apply(lambda col: col.map('{:.4g}'.format)).to_string())

                                                     Probit      Logit
Log-Likelihood                                       -5.497     -5.594
LL-Null                                              -42.98     -42.98
LLR (χ², 2 степени свободы)                           74.96      74.76
p-value LLR                                       5.292e-17  5.828e-17
Pseudo R² (McFadden)                                 0.8721     0.8698
AIC                                                   16.99      17.19
BIC                                                   23.38      23.57
Верно классифицировано, доля                         0.9677     0.9516
Верно классифицировано, шт. из 62                        60         59
Банкротов, признанных небанкротами                        1          1
Небанкротов, признанных банкротами                        1          2
ROC AUC                                              0.9938     0.9938
Brier score (средний квадрат ошибки вероятности)    0.02822    0.02793


In [17]:
# какие организации модели классифицируют неверно
for title, res in [('Probit', probit_model), ('Logit', logit_model)]:
    p = np.asarray(res.predict(X.astype(float)))
    wrong = data[(p > 0.5).astype(int) != Y.to_numpy()].copy()
    wrong['расчётная вероятность банкротства'] = p[wrong.index].round(3)
    print(title)
    print(wrong.to_string())
    print()

Probit
          Х1        Х2  Y  расчётная вероятность банкротства
23  1.396297  0.166123  0                              0.894
42  0.048192  0.558946  1                              0.363

Logit
          Х1        Х2  Y  расчётная вероятность банкротства
23  1.396297  0.166123  0                              0.910
29  3.857401  0.233876  0                              0.509
42  0.048192  0.558946  1                              0.379



**Выводы.**
- Все критерии согласия указывают на практически одинаковое качество моделей. Probit немного лучше: $LL = -5{,}497$ против $-5{,}594$, $AIC = 16{,}99$ против $17{,}19$, $BIC = 23{,}38$ против $23{,}57$, $R^2_{McFadden} = 0{,}8721$ против $0{,}8698$. Но разница $AIC$ составляет $0{,}19$, а различия менее 2 принято считать практически незначимыми, поэтому статистически выбрать одну из моделей нельзя.
- Обе модели значимы в целом (по тесту отношения правдоподобия $\chi^2 \approx 75$, $p \approx 5\cdot10^{-17}$), $ROC\ AUC$ у обеих $0{,}994$ (разделение классов почти идеальное).
- Доля верных классификаций: probit — 60 из 62 ($96{,}8\ \%$), logit — 59 из 62 ($95{,}2\ \%$). Разница — одна организация (№ 29, не банкрот): её расчётная вероятность банкротства $0{,}4945$ в probit и $0{,}5088$ в logit лежит по разные стороны порога $0{,}5$, то есть это следствие порога, а не качества модели. Организации 23 (не банкрот, расчётная вероятность банкротства $0{,}894$ в probit и $0{,}910$ в logit) и 42 (банкрот, $0{,}363$ и $0{,}379$) ошибочно классифицируются обеими моделями (см. таблицу выше).
- Все показатели рассчитаны по тем же 62 наблюдениям, на которых модели обучались, поэтому они оптимистичны; для проверки на новых данных нужна отложенная выборка или кросс-валидация. При $N = 62$ и почти полном разделении классов (см. раздел 3) такая проверка особенно желательна.
- Выбор между моделями определяется интерпретацией: probit соответствует методичке, logit даёт удобные отношения шансов.

## Итоги сверки с методичкой

| Что | В методичке | Реальное значение | Комментарий |
|---|---|---|---|
| Описательная статистика (рис. 2.15) | 62 организации, 31 / 31 | совпадает до 6 знака | расхождений нет |
| Probit: coef, std err, z, p, ДИ, LL, Pseudo R² (рис. 2.16) | см. рисунок | совпадает | расхождения только в дате и времени запуска |
| «Pseudo R² близок к 1» | 0,8721 | 0,8721 | значение верное; для моделей бинарного выбора это очень высокое значение (классы почти разделены) |
| Вероятность банкротства «средней» организации (рис. 2.17) | 26 % | **82,4 %** | в методичке посчитана плотность $\varphi(z)=0{,}2588$ вместо $\Phi(z)$; число $0{,}2588$ воспроизводится точно, но вероятностью не является |
| Предельный эффект $X_1$ (рис. 2.18) | $-0{,}0947$, «на 9,5 %» | $-0{,}0947$ | число верное; единица измерения — 9,5 **п.п.**, и это значение в точке средних, статистически незначимое ($p = 0{,}40$); по выборке в среднем $-0{,}0175$ ($p < 0{,}001$) |
| Примечание statsmodels под таблицей | на рис. 2.16 не показано | `Possibly complete quasi-separation` (probit 0,56; logit 0,42) | не ошибка: оценка сошлась и совпадает с методичкой; следствие почти полного разделения классов |